In [0]:
# ============================================================
# FASE 1B — Extract: Catálogo de géneros desde TMDB API
# ============================================================
import requests
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql import Row
from datetime import datetime

# ----------------------------------------------------------
# 1. Recuperar credenciales desde Databricks Secrets
# ----------------------------------------------------------
api_key = dbutils.secrets.get(scope="tmdb", key="api_key")
print(f"API key cargada: {api_key}")  # → [REDACTED]

# ----------------------------------------------------------
# 2. Llamada al endpoint de géneros
# ----------------------------------------------------------
response = requests.get(
    "https://api.themoviedb.org/3/genre/movie/list",
    params={
        "api_key":  api_key,
        "language": "it"   # nombres de género en italiano
    }
)

if response.status_code != 200:
    raise Exception(f"Error al llamar a la API: {response.status_code}")

generos_raw = response.json().get("genres", [])
print(f"Géneros obtenidos: {len(generos_raw)}")

# ----------------------------------------------------------
# 3. Parsear respuesta → filas
# ----------------------------------------------------------
filas = [
    Row(
        genero_id     = str(g["id"]),
        genero_nombre = g["name"]
    )
    for g in generos_raw
]

# ----------------------------------------------------------
# 4. Schema explícito y creación del DataFrame
# ----------------------------------------------------------
generos_schema = StructType([
    StructField("genero_id",     StringType(), False),
    StructField("genero_nombre", StringType(), False),
])

df_generos_raw = spark.createDataFrame(filas, schema=generos_schema)

df_generos_raw.show(truncate=False)
print(f"Total géneros: {df_generos_raw.count()}")

# ----------------------------------------------------------
# 5. Guardar en capa RAW (Unity Catalog)
# ----------------------------------------------------------
spark.sql("CREATE DATABASE IF NOT EXISTS etl_cine")

df_generos_raw.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("etl_cine.generos_raw")

print("✓ Tabla guardada: etl_cine.generos_raw")